# Fusion Analysis

## 1 Overview

## 2 Importing Libraries

In [2]:
from pathlib import Path
import sys

# Point Python at the project root so "src" can be imported.
# Adjust the number of .parent if needed (see note below).
project_root = Path.cwd()
while not (project_root / "src").exists() and project_root != project_root.parent:
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

import pandas as pd
import matplotlib.pyplot as plt

from src.ai.multimodal_pipeline import MultimodalPipeline

c:\Users\Subathra\OneDrive\Desktop\cm3020_Final_Year_project\CM3020_Final_Year_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3 Loading the Pipeline and Clips

In [ ]:
clips_folder = Path("data/raw/fusion_clips")

clip_paths = sorted(clips_folder.glob("*.mp4"))

print("Found", len(clip_paths), "clips")

pipeline = MultimodalPipeline()

## 4 Running Each Clip Through the Pipeline

In [ ]:
def silent_progress(message_key):
    # The pipeline expects a progress callback; not needed here.
    pass


rows = []

for path in clip_paths:
    result = pipeline.analyse(
        str(path),
        "video",
        silent_progress,
    )

    rows.append({
        "Clip": path.stem,
        "Text score": result["text_score"],
        "Audio score": result["audio_score"],
        "Vision score": result["vision_score"],
        "Fused wellbeing": result["wellbeing_score"],
    })

    print("Processed", path.name)

results_df = pd.DataFrame(rows)
results_df

## 5 Converting Scores to a Common Scale

In [ ]:
comparison_df = pd.DataFrame({
    "Clip": results_df["Clip"],
    "Text only": 100 - results_df["Text score"],
    "Audio only": 100 - results_df["Audio score"],
    "Vision only": 100 - results_df["Vision score"],
    "Fused": results_df["Fused wellbeing"],
})

comparison_df

## 6 Visual Comparison

In [ ]:
plot_df = comparison_df.set_index("Clip")

plot_df.plot(kind="bar", figsize=(12, 5))
plt.ylabel("Wellbeing scale (0-100)")
plt.title("Single modality vs fused wellbeing score per clip")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Score type")
plt.tight_layout()
plt.savefig("fusion_comparison.png")
plt.show()

## 7 Agreement Between Modalities

In [ ]:
singles = comparison_df[["Text only", "Audio only", "Vision only"]]

analysis_df = pd.DataFrame({
    "Clip": comparison_df["Clip"],
    "Lowest single": singles.min(axis=1).round(1),
    "Highest single": singles.max(axis=1).round(1),
    "Spread": (singles.max(axis=1) - singles.min(axis=1)).round(1),
    "Fused": comparison_df["Fused"].round(1),
})

analysis_df

## 8 Conclusion